**Import Libraries**

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Mount ADLS**

In [ ]:
storage_account = "inventorystaccount"
container = "adlsdestination"
access_key = "YOUR_ACCESS_KEY_HERE"

mount_point = "/mnt/inventory"

if not any(m.mountPoint == mount_point for m in dbutils.fs.mounts()):

    configs = {
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net": access_key
    }

    dbutils.fs.mount(
        source=f"wasbs://{container}@{storage_account}.blob.core.windows.net/",
        mount_point=mount_point,
        extra_configs=configs
    )

print("Mount Ready")

Mount Ready


In [ ]:
display(dbutils.fs.ls("/mnt/inventory"))

path,name,size,modificationTime
dbfs:/mnt/inventory/DIM.Date.Table.csv,DIM.Date.Table.csv,796835,1780981041000
dbfs:/mnt/inventory/Inventory_Management_Src1.csv,Inventory_Management_Src1.csv,287957,1780981040000


**Read Files**

In [ ]:
inventory_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/mnt/inventory/Inventory_Management_Src1.csv")

date_df = spark.read \
.option("header","true") \
.option("inferSchema","true") \
.csv("/mnt/inventory/DIM.Date.Table.csv")

Remove Extra Spaces From Column Names

In [ ]:
inventory_df = inventory_df.toDF(
    *[c.strip() for c in inventory_df.columns]
)

Create Date Dimension

In [ ]:
dim_date = date_df.select(
    "date_key",
    "full_date",
    "month_name",
    "quarter",
    "fb_year"
)

In [ ]:
print(inventory_df.columns)

['Date', 'TRANSACTION_ID', 'TRANSACTION_DATE', 'TRANSACTION_AMOUNT', 'TRANSACTION_TYPE', 'DISPATCH_DATE', 'EXPECTED_DATE', 'DELIVERY_DATE', 'PRODUCT_ID', 'CATEGORY_ID', 'PRODUCT Name', 'PRODUCT Brand', 'Product Model No.', 'PRODUCT_STOCK', 'CUSTOMER_ID', 'Customer Name', 'CUSTOMER_LOGIN_ID', 'CUSTOMER_STREET_ADDRESS', 'CUSTOMER_CITY', 'CUSTOMER_STATE', 'CUSTOMER_ZIP', 'CUSTOMER_PHONE_NO', 'SELLER_ID', 'SELLER_NAME', 'SELLER_RATING', 'SELLER_STREET_ADDRESS', 'SELLER_CITY', 'SELLER_STATE', 'SELLER_ZIP', 'SELLER_PHONE_NO', 'PRODUCT_COST_PRICE', 'PRODUCT_SELLING_PRICE']


Create Product Dimension

In [ ]:
dim_product = inventory_df.select(
    "PRODUCT_ID",
    "CATEGORY_ID",
    "PRODUCT Name",
    "PRODUCT Brand",
    "Product Model No.",
    "PRODUCT_COST_PRICE",
    "PRODUCT_SELLING_PRICE"
).dropDuplicates()

dim_product = dim_product \
.withColumnRenamed("PRODUCT Name","PRODUCT_NAME") \
.withColumnRenamed("PRODUCT Brand","PRODUCT_BRAND") \
.withColumnRenamed("Product Model No.","PRODUCT_MODEL_NO")

Create Customer Dimension

In [ ]:
dim_customer = inventory_df.select(
    "CUSTOMER_ID",
    "Customer Name",
    "CUSTOMER_CITY",
    "CUSTOMER_STATE",
    "CUSTOMER_ZIP",
    "CUSTOMER_PHONE_NO"
).dropDuplicates()

dim_customer = dim_customer \
.withColumnRenamed("Customer Name","CUSTOMER_NAME")

Create Seller Dimension

In [ ]:
dim_seller = inventory_df.select(
    "SELLER_ID",
    "SELLER_NAME",
    "SELLER_RATING",
    "SELLER_CITY",
    "SELLER_STATE",
    "SELLER_ZIP",
    "SELLER_PHONE_NO"
).dropDuplicates()

Create Date Key

In [ ]:
from pyspark.sql.functions import *

inventory_df = inventory_df.withColumn(
    "date_key",
    date_format(
        to_date(col("Date"), "d-MMM-yy"),
        "yyyyMMdd"
    ).cast("int")
)

Create Fact Table

In [ ]:
fact_inventory = inventory_df.select(
    "TRANSACTION_ID",
    "TRANSACTION_DATE",
    "TRANSACTION_AMOUNT",
    "TRANSACTION_TYPE",
    "PRODUCT_ID",
    "CATEGORY_ID",
    "CUSTOMER_ID",
    "SELLER_ID",
    "PRODUCT_STOCK",
    "PRODUCT_COST_PRICE",
    "PRODUCT_SELLING_PRICE",
    "date_key"
)

**Save Dimension Tables as Delta**

Dim Date

In [ ]:
dim_date.write \
.format("delta") \
.mode("overwrite") \
.save("/mnt/inventory/delta/dim_date")

Dim Product

In [ ]:
dim_product.write \
.format("delta") \
.mode("overwrite") \
.save("/mnt/inventory/delta/dim_product")

Dim Customer

In [ ]:
dim_customer.write \
.format("delta") \
.mode("overwrite") \
.save("/mnt/inventory/delta/dim_customer")

Dim Seller

In [ ]:
dim_seller.write \
.format("delta") \
.mode("overwrite") \
.save("/mnt/inventory/delta/dim_seller")

Fact Inventory

In [ ]:
fact_inventory.write \
.format("delta") \
.mode("overwrite") \
.save("/mnt/inventory/delta/fact_inventory")

**Register Delta Native Tables**

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS dim_date
USING DELTA
LOCATION '/mnt/inventory/delta/dim_date';

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS dim_product
USING DELTA
LOCATION '/mnt/inventory/delta/dim_product';

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS dim_customer
USING DELTA
LOCATION '/mnt/inventory/delta/dim_customer';

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS dim_seller
USING DELTA
LOCATION '/mnt/inventory/delta/dim_seller';

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS fact_inventory
USING DELTA
LOCATION '/mnt/inventory/delta/fact_inventory';

In [ ]:
fact_report = fact_inventory.join(
    dim_date,
    "date_key",
    "left"
)

**Generate Report 1: Product Sales**

In [ ]:
from pyspark.sql.functions import *

product_sales = inventory_df.groupBy(
    "PRODUCT Name"
).agg(
    sum("TRANSACTION_AMOUNT").alias("TOTAL_SALES")
)

display(product_sales)

PRODUCT Name,TOTAL_SALES
SHOES,954620
CAMERA,718270
SUNGLASSES,649720
EARPHONES,978060
DVD PLAYER,1434400
PENDRIVE,810170
TELEVISION,1264580
BAG,1270820
PRINTER,563370
WATCH,521960


In [ ]:
product_sales.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/mnt/inventory/reports/product_sales")

**Generate Report 2: Monthly Sales**

In [ ]:
monthly_sales = fact_report.groupBy(
    "month_name"
).agg(
    sum("TRANSACTION_AMOUNT").alias("TOTAL_SALES")
)

display(monthly_sales)

month_name,TOTAL_SALES
July,1000000
November,664530
February,941740
January,1004660
March,889390
null,1550
October,481100
May,1067610
August,928150
April,936680


In [ ]:
monthly_sales.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/mnt/inventory/reports/monthly_sales")

**Generate Report 3: Seller Revenue**

In [ ]:
seller_revenue = inventory_df.groupBy(
    "SELLER_NAME"
).agg(
    sum("TRANSACTION_AMOUNT").alias("TOTAL_REVENUE")
)

display(seller_revenue)

SELLER_NAME,TOTAL_REVENUE
Dahlia Moss,146500
Ciaran Butler,92700
Rhonda Daniels,50150
Rose Page,114660
Chelsea Nguyen,152200
Bo Curry,16900
Chaim Riley,114800
Moana Alexander,67300
Nathaniel Mcbride,84800
Eve Coffey,48750


In [ ]:
seller_revenue.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/mnt/inventory/reports/seller_revenue")

**Generate Report 4: State-wise Sales**

In [ ]:
state_sales = inventory_df.groupBy(
    "CUSTOMER_STATE"
).agg(
    sum("TRANSACTION_AMOUNT").alias("TOTAL_SALES")
)

display(state_sales)

CUSTOMER_STATE,TOTAL_SALES
PENNSYLVANIA,665800
ILLINOIS,632850
DISTRICT OF COLUMBIA,547820
NEWYORK,666650
TEXAS,2087270
CALIFORNIA,1937560
ARIZONA,741880
MASSAACHUSETTS,943910
OHIO,1138270
California,910290


In [ ]:
state_sales.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/mnt/inventory/reports/state_sales")

**Generate Report 5: Top 10 Products**

In [ ]:
top10_products = inventory_df.groupBy(
    "PRODUCT Name"
).agg(
    sum("TRANSACTION_AMOUNT").alias("TOTAL_REVENUE")
).orderBy(
    desc("TOTAL_REVENUE")
).limit(10)

display(top10_products)

PRODUCT Name,TOTAL_REVENUE
DVD PLAYER,1434400
BAG,1270820
TELEVISION,1264580
LAPTOP,1106330
EARPHONES,978060
SHOES,954620
PENDRIVE,810170
CAMERA,718270
SUNGLASSES,649720
PRINTER,563370


In [ ]:
top10_products.coalesce(1).write \
.mode("overwrite") \
.option("header", "true") \
.csv("/mnt/inventory/reports/top10_products")